# Basic RAG Starter

A minimal RAG baseline using minsearch, following the same structure we used in the module: a `search` function, a `build_prompt` function, an `llm` function, and a top-level `rag` function that chains them together.

Replace the example data with your own and adapt the instructions to match what your project should actually do.

In [ ]:
import json

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from minsearch import Index

openai_client = OpenAI()

## 1. Load Your Data

Replace this example with your own data. Each document should be a dict with a few text fields you want to search over.

In [ ]:
documents = [
    {"title": "Getting Started", "content": "Install uv, create a .env file, and run uv sync to install dependencies."},
    {"title": "Running Notebooks", "content": "Use uv run jupyter notebook to start Jupyter and open a notebook."},
    {"title": "Environment Variables", "content": "Keep your API keys in .env. The file is gitignored so it never ends up in your repo."},
    {"title": "Adding Dependencies", "content": "Use uv add PACKAGE_NAME to add a new dependency to pyproject.toml."},
    {"title": "Basic RAG", "content": "RAG means retrieving relevant documents from an index and passing them to the LLM as context."},
]

## 2. Build the Index

minsearch builds an in-memory TF-IDF index over the fields you specify.

In [ ]:
index = Index(
    text_fields=["title", "content"],
    keyword_fields=[],
)

index.fit(documents)

## 3. Define the RAG Functions

`search` retrieves relevant documents, `build_prompt` formats them into a prompt, `llm` calls the model, and `rag` chains all three.

In [ ]:
def search(query):
    return index.search(query, num_results=5)

In [ ]:
instructions = """
You're a helpful assistant. Answer the QUESTION using only the information
in the CONTEXT below. If the context doesn't contain the answer, say you
don't know.
""".strip()


def build_prompt(query, search_results):
    search_result_json = json.dumps(search_results, indent=2)

    user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

    return user_prompt

In [ ]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [ ]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

## 4. Try It Out

In [ ]:
print(rag("How do I add a new dependency?"))